# 11. Combined Sidewalk Width

Panoramas and LiDAR point clouds are captured on different dates, so a
single-snapshot comparison is unreliable — a bike seen in today's panorama
may not be in last year's point cloud, or vice versa. This notebook produces
two usable-width numbers per sidewalk segment instead of one:

- **Structural width** — BGT voetpad polygon minus *permanent* LiDAR
  obstacles only (trees, benches, bollards, bike racks, poles). This doesn't
  depend on when the point cloud was captured, since permanent things don't
  move.
- **Typical/observed width** — structural width additionally reduced by
  *temporary* obstacles (parked bikes, scooters, cars) detected in panorama
  imagery across **several `mission_year` epochs**. Per cross-section point,
  the *median* usable width across epochs is used, so a bike parked in one
  out of several years doesn't skew the result, but a spot that's typically
  cluttered does.

Requires notebook 4 (cluster inventory) and a BGT-covered tile. Panoramas are
fetched fresh per year (not reused from notebook 8's single-year cache).

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, str(Path.cwd()))
import config
from utils.labels import Labels
from utils.panorama_api import fetch_panoramas_in_tile, download_panorama_image
from utils.panorama_geometry import (
    CameraPose, make_patches, equirect_to_perspective,
    detection_to_ray, ray_ground_intersect,
)
from utils.detection import load_detector, detect_patches, detections_to_geodataframe
from utils.triangulation import triangulate_detections, correct_camera_elevation
from utils.ahn_reader import PolygonNPZReader
from utils.sidewalk_width import (
    fetch_voetpad_polygons, obstacle_footprints_from_inventory,
    panorama_obstacle_footprints, compute_usable_width,
    compute_typical_observed_width, width_to_geojson, width_category,
)

## Configuration

In [ ]:
TILECODE = config.SETUP_TILECODES[0]

# Panorama coverage years to sample. Confirm actual years with coverage for
# your area via the panorama API before relying on this list.
PANO_YEARS = [2021, 2022, 2023, 2024]

PANO_DIR       = Path("data/input/panoramas")
YOLO_WEIGHTS   = "yolov8m.pt"
DETECTION_CONF = 0.35
CAM_HEIGHT_M   = 2.0    # Cyclomedia camera is ~2 m above the road surface
MIN_SPACING_M  = 8.0    # minimum distance between kept panoramas per epoch
BUFFER_M       = 30.0   # grow the tile footprint when searching for cameras
IMAGE_SIZE     = "medium"
FORCE_REFETCH  = False

MATCH_RADIUS_M = 2.5    # cross-camera association radius (triangulation)
MIN_BASELINE_M = 3.0    # min camera separation to attempt triangulation
MAX_RESIDUAL_M = 1.5    # max RMS ray residual to accept a triangulation

N_PATCHES  = 4           # 4 x 90 deg = full 360 deg
FOV_H      = 90.0
PATCH_SIZE = (640, 640)

STEP_M             = 0.5    # cross-section interval
OBSTACLE_BUFFER_M  = 0.15   # buffer around LiDAR obstacle footprints
PANO_BUFFER_M      = 0.3    # buffer around panorama detection footprints

print(f"Tile: {TILECODE}")
print(f"Panorama epochs: {PANO_YEARS}")

## Step 1 — Fetch panoramas per epoch

Unlike notebook 8 (single `mission_year`, cached to `panoramas.json`), this
fetches and caches one file per year (`panoramas_<year>.json`) so several
epochs of the same tile coexist on disk.

In [ ]:
tile_pano_dir = PANO_DIR / TILECODE
tile_pano_dir.mkdir(parents=True, exist_ok=True)

PANORAMAS_BY_YEAR = {}

for year in PANO_YEARS:
    cache_path = tile_pano_dir / f"panoramas_{year}.json"

    if cache_path.exists() and not FORCE_REFETCH:
        with open(cache_path) as f:
            poses = json.load(f)
        print(f"  {year}: {len(poses)} panoramas (cached)")
    else:
        poses = fetch_panoramas_in_tile(
            TILECODE, bbox_dir=config.BBOX_DIR, mission_year=year,
            min_spacing_m=MIN_SPACING_M, buffer_m=BUFFER_M,
        )
        if poses:
            for pose in poses:
                download_panorama_image(pose, size=IMAGE_SIZE, cache_dir=tile_pano_dir)
            with open(cache_path, "w") as f:
                json.dump(poses, f, indent=2)
        print(f"  {year}: {len(poses)} panoramas fetched")

    PANORAMAS_BY_YEAR[year] = poses

n_epochs_with_data = sum(1 for p in PANORAMAS_BY_YEAR.values() if p)
print(f"\n{n_epochs_with_data}/{len(PANO_YEARS)} epochs have panorama coverage for {TILECODE}.")

## Step 2 — Detect + triangulate per epoch

Runs the same multi-camera detect -> AHN ground correction -> triangulation
flow as notebook 9's "Multi-camera detection map" cell, once per epoch, and
exports one GeoJSON per year.

In [ ]:
model = load_detector(YOLO_WEIGHTS)
ahn_reader = PolygonNPZReader(config.AHN_NPZ_DIR)

out_dir = config.CLUSTERS_DIR / TILECODE
out_dir.mkdir(parents=True, exist_ok=True)

CAM_RESULTS_BY_YEAR = {}

for year, poses in PANORAMAS_BY_YEAR.items():
    if not poses:
        CAM_RESULTS_BY_YEAR[year] = []
        continue

    cam_results = []
    for pose_d in poses:
        cam_pose = CameraPose(
            x_rd=pose_d["x_rd"], y_rd=pose_d["y_rd"], z=pose_d["z_wgs84"],
            heading=pose_d["heading"], pitch=pose_d["pitch"],
            roll=pose_d.get("roll", 0.0),
        )
        img_p = tile_pano_dir / f"{pose_d['pano_id']}.jpg"
        if not img_p.exists():
            continue

        img_i = np.array(Image.open(img_p).convert("RGB"))
        pps = make_patches(
            heading_deg=cam_pose.heading, fov_h_deg=FOV_H,
            out_hw=PATCH_SIZE, n_horizontal=N_PATCHES,
            pitch_deg=0.0, img_shape=img_i.shape,
        )
        patches_i = [equirect_to_perspective(img_i, pp) for pp in pps]
        dets_i    = detect_patches(patches_i, pps, model, conf=DETECTION_CONF)

        correct_camera_elevation(cam_pose, ahn_reader, TILECODE, cam_height_m=CAM_HEIGHT_M)

        gz = cam_pose.z - CAM_HEIGHT_M
        for det in dets_i:
            x1, y1, x2, y2 = det.bbox_xyxy
            origin, ray = detection_to_ray((x1, y2, x2, y2), pps[det.patch_idx], cam_pose)
            det.ray_origin = tuple(origin)
            det.ray_dir    = tuple(ray)
            result = ray_ground_intersect(cam_pose, ray, gz)
            if result is not None:
                det.x_rd, det.y_rd = result
                det.x_rd_single, det.y_rd_single = result

        valid = [d for d in dets_i if d.x_rd is not None]
        cam_results.append((pose_d, cam_pose, valid))

    if cam_results:
        triangulate_detections(
            cam_results, TILECODE, ahn_reader,
            match_radius_m=MATCH_RADIUS_M, min_baseline_m=MIN_BASELINE_M,
            max_residual_m=MAX_RESIDUAL_M, cam_height_m=CAM_HEIGHT_M,
        )

    CAM_RESULTS_BY_YEAR[year] = cam_results

    all_dets = [d for _, _, dets in cam_results for d in dets]
    print(f"  {year}: {len(cam_results)} cameras, {len(all_dets)} detections", end="")
    if all_dets:
        out_path = out_dir / f"panorama_detections_{TILECODE}_{year}.geojson"
        gdf = detections_to_geodataframe(all_dets)
        gdf.to_crs("EPSG:4326").to_file(str(out_path), driver="GeoJSON")
        print(f"  -> {out_path}")
    else:
        print()

## Step 3 — BGT voetpad polygons + permanent (structural) LiDAR obstacles

In [ ]:
BBOX_BUFFER_M = 30.0
bbox_path = config.BBOX_DIR / f"bbox_{TILECODE}.geojson"
with open(bbox_path) as f:
    gj = json.load(f)
coords = np.array(gj["features"][0]["geometry"]["coordinates"][0])
TILE_BBOX_RD = (
    float(coords[:, 0].min()) - BBOX_BUFFER_M,
    float(coords[:, 1].min()) - BBOX_BUFFER_M,
    float(coords[:, 0].max()) + BBOX_BUFFER_M,
    float(coords[:, 1].max()) + BBOX_BUFFER_M,
)

voetpad_gdf = fetch_voetpad_polygons(TILE_BBOX_RD)
print(f"{len(voetpad_gdf)} voetpad polygons.")

inv_path = config.CLUSTERS_DIR / f"inventory_{TILECODE}.csv"
if not inv_path.exists():
    inv_path = config.CLUSTERS_DIR / TILECODE / f"inventory_{TILECODE}.csv"
cluster_df = pd.read_csv(inv_path)

permanent_gdf = obstacle_footprints_from_inventory(
    cluster_df, clusters_dir=config.CLUSTERS_DIR / TILECODE,
    buffer_m=OBSTACLE_BUFFER_M, permanence='permanent',
)
print(f"{len(permanent_gdf)} permanent (structural) obstacle footprints.")
print("Labels:", {Labels.STR_DICT.get(int(k), str(k)): v
                  for k, v in permanent_gdf["label"].value_counts().items()})

## Step 4 — Structural width (permanent obstacles only)

In [ ]:
structural_width_gdf = compute_usable_width(voetpad_gdf, permanent_gdf, step_m=STEP_M)
structural_width_gdf["category"] = structural_width_gdf["width_usable_m"].apply(width_category)
print(f"{len(structural_width_gdf)} measurement segments.")
print(structural_width_gdf[["width_total_m", "width_usable_m"]].describe())

## Step 5 — Typical/observed width (permanent + median temporary clutter)

Loads each epoch's exported panorama detections, keeps only
*temporary*-classed obstacles (bikes, scooters, cars — via the same
`_BGT_PERMANENCE` taxonomy used for the LiDAR side), and combines them with
the permanent layer per epoch before taking the per-point median.

In [ ]:
epoch_obstacles = []
for year in PANO_YEARS:
    det_path = out_dir / f"panorama_detections_{TILECODE}_{year}.geojson"
    if not det_path.exists():
        continue
    det_gdf = gpd.read_file(det_path).to_crs("EPSG:28992")
    temp_gdf = panorama_obstacle_footprints(
        det_gdf, permanence='temporary', buffer_m=PANO_BUFFER_M)
    print(f"  {year}: {len(temp_gdf)} temporary obstacle footprints")
    epoch_obstacles.append(temp_gdf)

typical_width_gdf = compute_typical_observed_width(
    voetpad_gdf, permanent_gdf, epoch_obstacles, step_m=STEP_M)
typical_width_gdf["category"] = typical_width_gdf["width_usable_typical_m"].apply(width_category)
print(f"\n{len(typical_width_gdf)} measurement segments, {typical_width_gdf['n_epochs'].iloc[0] if len(typical_width_gdf) else 0} epochs.")
print(typical_width_gdf[["width_total_m", "width_usable_typical_m", "n_epochs_clutter"]].describe())

## Step 6 — Compare structural vs. typical width

In [ ]:
comparison = structural_width_gdf[["voetpad_idx", "dist_m", "width_total_m", "width_usable_m", "category"]].merge(
    typical_width_gdf[["voetpad_idx", "dist_m", "width_usable_typical_m", "n_epochs_clutter",
                        "category"]].rename(columns={"category": "category_typical"}),
    on=["voetpad_idx", "dist_m"],
)
comparison["delta_m"] = comparison["width_usable_m"] - comparison["width_usable_typical_m"]

print("Category counts — structural:")
print(comparison["category"].value_counts().to_string())
print("\nCategory counts — typical/observed:")
print(comparison["category_typical"].value_counts().to_string())

n_recurring = int((comparison["n_epochs_clutter"] >= 2).sum())
n_downgraded = int((comparison["category"] != comparison["category_typical"]).sum())
print(f"\nSegments with recurring clutter (>=2 epochs): {n_recurring}/{len(comparison)}")
print(f"Segments whose accessibility category changed once clutter is included: {n_downgraded}/{len(comparison)}")

In [ ]:
CATEGORY_COLORS = {
    "good": "#2ecc71", "adequate": "#f39c12",
    "narrow": "#e74c3c", "blocked": "#8e44ad", "unknown": "#95a5a6",
}

fig, axes = plt.subplots(1, 2, figsize=(15, 7.5))

for ax, gdf, width_col, cat_col, title in [
    (axes[0], structural_width_gdf, "width_usable_m", "category", "Structural width\n(permanent LiDAR obstacles only)"),
    (axes[1], typical_width_gdf, "width_usable_typical_m", "category", f"Typical/observed width\n(median across {len(PANO_YEARS)} panorama epochs)"),
]:
    voetpad_gdf.plot(ax=ax, color="#ecf0f1", edgecolor="#bdc3c7", zorder=1)
    for cat, color in CATEGORY_COLORS.items():
        sub = gdf[gdf[cat_col] == cat]
        if len(sub):
            sub.plot(ax=ax, color=color, linewidth=2.5, label=cat, zorder=2)
    ax.set_aspect("equal")
    ax.legend(title="Usable width", loc="upper right", fontsize=8)
    ax.set_title(f"{title}\ntile {TILECODE}", fontsize=10)
    ax.set_xlabel("RD New X"); ax.set_ylabel("RD New Y")

plt.tight_layout()
plt.show()

## Step 7 — Export

In [ ]:
structural_out = out_dir / f"sidewalk_width_structural_{TILECODE}.geojson"
typical_out    = out_dir / f"sidewalk_width_typical_{TILECODE}.geojson"

width_to_geojson(structural_width_gdf, structural_out)
width_to_geojson(typical_width_gdf.drop(columns=["n_epochs"]), typical_out)